# FedMed-BN: Notebook 1 - Setup & Data Exploration
**Privacy-Preserving Federated Learning for Bangla Medical Text NER**

Run this notebook in **Google Colab with GPU** (Runtime > Change runtime type > T4 GPU)

In [ ]:
# Cell 1: Install all required libraries
!pip install transformers datasets torch flower-federated opacus -q
!pip install bnlp-toolkit indic-nlp-library seqeval -q
!pip install scikit-learn pandas matplotlib seaborn -q

print("Libraries installed successfully!")

In [ ]:
# Cell 2: Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch version: {torch.__version__}")

# You NEED GPU runtime. Go to Runtime > Change runtime type > T4 GPU

In [ ]:
# Cell 3: Mount Google Drive (RECOMMENDED - persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Put datasets in /content/drive/MyDrive/FedMed-BN/data/
# Or copy from local upload:
import os
os.makedirs('/content/FedMed-BN/data', exist_ok=True)
!cp -r /content/drive/MyDrive/FedMed-BN/data/* /content/FedMed-BN/data/ 2>/dev/null || echo "Copy from Drive or upload manually"

In [ ]:
# Cell 4: Load and explore the preprocessed data
import os

def load_bio_file(filepath):
    # Load CoNLL format BIO file
    sentences = []
    current = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current.append((parts[0], parts[1]))
    if current:
        sentences.append(current)
    return sentences

data_dir = '/content/FedMed-BN/data'

# Load all datasets
centralized_train = load_bio_file(f'{data_dir}/centralized_train.txt')
centralized_test = load_bio_file(f'{data_dir}/centralized_test.txt')

client_data = {}
for i in range(3):
    client_data[i] = {
        'train': load_bio_file(f'{data_dir}/client_{i}_train.txt'),
        'test': load_bio_file(f'{data_dir}/client_{i}_test.txt')
    }

print(f"Centralized train: {len(centralized_train)} sentences")
print(f"Centralized test: {len(centralized_test)} sentences")
for i in range(3):
    print(f"Client {i}: train={len(client_data[i]['train'])}, test={len(client_data[i]['test'])}")

In [ ]:
# Cell 5: Analyze entity distribution
from collections import Counter

def get_entity_stats(sentences):
    entity_counts = Counter()
    token_count = 0
    for sent in sentences:
        for token, tag in sent:
            token_count += 1
            if tag != 'O':
                entity_counts[tag] += 1
    return entity_counts, token_count

print("=== CENTRALIZED TRAIN ===")
ent_counts, tokens = get_entity_stats(centralized_train)
print(f"Tokens: {tokens}")
for tag, count in ent_counts.most_common():
    print(f"  {tag}: {count}")

print("\n=== PER CLIENT TRAIN ===")
for i in range(3):
    print(f"\nClient {i}:")
    ent_counts, tokens = get_entity_stats(client_data[i]['train'])
    print(f"  Tokens: {tokens}")
    for tag, count in ent_counts.most_common():
        print(f"  {tag}: {count}")

In [ ]:
# Cell 6: Show sample sentences
print("=== SAMPLE SENTENCES ===")
for i, sent in enumerate(centralized_train[:3]):
    print(f"\nSentence {i+1}:")
    for token, tag in sent:
        if tag != 'O':
            print(f"  {token} -> {tag}")
        else:
            print(f"  {token}")

In [ ]:
# Cell 7: Build label vocabulary
all_tags = set()
for sent in centralized_train:
    for _, tag in sent:
        all_tags.add(tag)

# Sort: O first, then B-*, then I-*
tag_list = ['O']
b_tags = sorted([t for t in all_tags if t.startswith('B-')])
i_tags = sorted([t for t in all_tags if t.startswith('I-')])
tag_list.extend(b_tags)
tag_list.extend(i_tags)

tag2id = {tag: i for i, tag in enumerate(tag_list)}
id2tag = {i: tag for i, tag in enumerate(tag_list)}

print(f"Total labels: {len(tag_list)}")
for tag, idx in tag2id.items():
    print(f"  {idx}: {tag}")

# Save for later use
import json
with open('/content/FedMed-BN/data/tag2id.json', 'w') as f:
    json.dump(tag2id, f)
with open('/content/FedMed-BN/data/id2tag.json', 'w') as f:
    json.dump(id2tag, f)

print("\nLabel mappings saved!")